In [ ]:
['start_xy', 'end_xy', 'start_yaw', 'end_yaw', 'start_speed', 'end_speed', 'avg_speed', 'direction', 'turn_label', 'lane_change']

In [2]:
from __future__ import annotations

import json
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from tqdm import tqdm
import os

IMG_ROOT = Path('/zfsauton/scratch/eshau/imgs')
INSTRUCTION_ROOT = Path('/zfsauton/scratch/mineuih/waymax_rs/annotations/')
TARGET_ROOT = Path('/zfsauton/scratch/mineuih/waymax_rs/manual_instruction/')
os.makedirs(TARGET_ROOT, exist_ok=True)


def write_jsonl_with_byte_index(records, output_path: Path):
    byte_offsets = []
    with output_path.open('wb') as out_f:
        for record in records:
            byte_offsets.append(out_f.tell())
            rec_bytes = (json.dumps(record, ensure_ascii=False) + '\n').encode('utf-8')
            out_f.write(rec_bytes)

    index_path = output_path.with_suffix('.idx.json')
    with index_path.open('w', encoding='utf-8') as idx_f:
        json.dump(
            {
                'source_file': output_path.name,
                'byte_offsets': byte_offsets,
            },
            idx_f,
            ensure_ascii=False,
            indent=2,
        )


for file_idx in range(1000):
    for timestep in [10]:
        file_path = INSTRUCTION_ROOT / f"training_tfexample.tfrecord-{file_idx:05d}-of-01000_t{timestep}.jsonl"
        with open(file_path, 'r', encoding='utf-8') as f:
            records = []
            scenario_index = 0
            for i, item in tqdm(enumerate(f), desc=f"File {file_idx:05d}"):
                record = json.loads(item)
                annot = record['annotation']['ego_motion']
                if scenario_index != record['scenario_index']:
                    # raise ValueError(f"Scenario index mismatch: {scenario_index} vs {record['scenario_index']}")
                    continue
                # speed instruction
                if float(annot['start_speed']) < 0.1:
                    if float(annot['end_speed']) < 0.1:
                        speed_inst = 'stop'
                    elif float(annot['end_speed']) > 0.1:
                        speed_inst = 'stop and go'
                elif np.abs(float(annot['start_speed']) - float(annot['end_speed'])) / float(annot['start_speed']) < 0.1:
                    speed_inst = 'maintain'
                elif float(annot['start_speed']) < float(annot['end_speed']):
                    speed_inst = 'accelerate'
                else:
                    speed_inst = 'decelerate'
                lane_is_known = True
                if speed_inst == 'stop':
                    instruction = 'stop'
                else:
                    instruction = ''
                    if speed_inst == 'stop and go':
                        instruction += 'stop for a while, and then '

                    if annot['direction'] == 'straight':
                        instruction += 'go straight '
                    elif annot['direction'] == 'left':
                        instruction += 'go left '
                    elif annot['direction'] == 'right':
                        instruction += 'go right '
                    elif annot['direction'] == 'slight left':
                        instruction += 'go slightly left '
                    elif annot['direction'] == 'slight right':
                        instruction += 'go slightly right '
                    else:
                        instruction += 'go '

                    if 'left' in annot['turn_label']:
                        instruction += 'to turn left '
                    elif 'right' in annot['turn_label']:
                        instruction += 'to turn right '
                    elif 'u-turn' in annot['turn_label']:
                        instruction += 'to make a U-turn '
                    
                    if 'left' in annot['lane_change']:
                        instruction += 'while changing lane to the left '
                    elif 'right' in annot['lane_change']:
                        instruction += 'while changing lane to the right '
                    elif 'none' in annot['lane_change']:
                        instruction += 'while following current lane '
                    else:
                        lane_is_known = False
                    
                    if speed_inst == 'accelerate':
                        instruction += 'and accelerating' if lane_is_known else 'while accelerating'
                    elif speed_inst == 'decelerate':
                        instruction += 'and slowing down' if lane_is_known else 'while slowing down'
                record['instruction'] = instruction
                records.append(record)

                scenario_index += 1

        output_path = TARGET_ROOT / f"training_tfexample.tfrecord-{file_idx:05d}-of-01000_t{timestep}.jsonl"
        write_jsonl_with_byte_index(records, output_path)


File 00000: 455it [00:00, 23126.62it/s]


File 00001: 479it [00:00, 16971.23it/s]
File 00002: 514it [00:00, 16855.66it/s]
File 00003: 479it [00:00, 17123.40it/s]
File 00004: 495it [00:00, 17248.04it/s]
File 00005: 465it [00:00, 16889.67it/s]
File 00006: 516it [00:00, 18956.31it/s]
File 00007: 468it [00:00, 6256.54it/s]
File 00008: 499it [00:00, 16789.33it/s]
File 00009: 481it [00:00, 20429.98it/s]
File 00010: 476it [00:00, 10326.90it/s]
File 00011: 501it [00:00, 16861.08it/s]
File 00012: 509it [00:00, 12922.42it/s]
File 00013: 476it [00:00, 15701.47it/s]
File 00014: 484it [00:00, 16072.80it/s]
File 00015: 494it [00:00, 18582.84it/s]
File 00016: 468it [00:00, 15984.94it/s]
File 00017: 453it [00:00, 16367.63it/s]
File 00018: 465it [00:00, 6900.26it/s]
File 00019: 487it [00:00, 14774.55it/s]
File 00020: 476it [00:00, 13373.76it/s]
File 00021: 450it [00:00, 6853.11it/s]
File 00022: 469it [00:00, 19452.45it/s]
File 00023: 478it [00:00, 20880.88it/s]
File 00024: 463it [00:00, 37275.19it/s]
File 00025: 499it [00:00, 7895.06it/s]
Fil

In [3]:
for file_path in TARGET_ROOT.glob("*.jsonl"):
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            record = json.loads(line)
            if i != record["scenario_index"]:
                raise ValueError(f"Scenario index mismatch: {i} vs {record['scenario_index']}")

In [18]:
from __future__ import annotations

import json
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from tqdm import tqdm
import os

IMG_ROOT = Path('/zfsauton/scratch/eshau/imgs')
INSTRUCTION_ROOT = Path('/zfsauton/scratch/mineuih/waymax_rs/new_annotations/')
TARGET_ROOT = Path('/zfsauton/scratch/mineuih/waymax_rs/manual_instruction/')
file_path = TARGET_ROOT / f"training_tfexample.tfrecord-00000-of-01000_t0.jsonl"
with open(file_path, 'r', encoding='utf-8') as f:
    for i, item in enumerate(f):
        if i % 10 == 0:
            record = json.loads(item)
            # print(json.dumps(record, indent=2, ensure_ascii=False))
            print(record['annotation']['lane_context']['traffic_lights'])
            # break

{}
{}
{}
{'straight': 'go'}
{}
{'straight': 'stop'}
{}
{'straight': 'stop', 'left': 'stop'}
{}
{'right': 'go'}
{'straight': 'stop'}
{}
{}
{}
{'left': 'stop', 'straight': 'stop'}
{}
{'straight': 'stop'}
{'right': 'stop'}
{}
{}
{'straight': 'stop'}
{}
{'straight': 'caution'}
{'straight': 'stop', 'right': 'stop', 'left': 'stop'}
{'right': 'arrow_stop', 'left': 'arrow_stop'}
[]
{}
{}
{'straight': 'go'}
{}
{'straight': 'stop'}
{}
{}
{}
{}
{'straight': 'stop'}
{'left': 'go', 'straight': 'go'}
{}
{'right': 'arrow_stop'}
{}
{}
{}
{'straight': 'stop'}
{'right': 'stop'}
{'straight': 'stop'}
{}


In [16]:
x = -0.01
print(f"{float(f'{x:.1f}')}")

-0.0
